# INT4 Quantization

In this notebook I am going to see the performance of the Quantization of INT4 (4GB). This model is where we are going to see really LLMs compressed.

The objectives of this notebook is analyzing:

    - How much memory do I save?
    - How much do I loss in quality?
    - Do I win velocity?



## Imports

In [1]:
import time
import gc
import torch
import psutil
import pandas as pd

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

## Config

In [4]:
MODEL_ID = "microsoft/phi-2"

PROMPT = """
You are an AI expert.

Explain quantization in large language models
for a beginner.

Use simple language and avoid code.
"""

MAX_NEW_TOKENS = 150

## Device

We are going to use only the CPU because BitsAndBytes with Mac and evit errors with MPS

In [5]:
device = torch.device("cpu")

print("Using CPU for INT4 inference")

Using CPU for INT4 inference


## RAM Helper

In [6]:
def get_ram_usage_gb():
    process = psutil.Process()
    memory_info = process.memory_info()
    return memory_info.rss / (1024 ** 3)

## Cleanup Helper

In [7]:
def cleanup():
    gc.collect()

## Load Tokenizer

In [8]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

## INT4 Config

In [9]:
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)

The bnb_4bit_compute_dtype is used to make some internal operations using FP16 ensuring the stability of the model

## Load Model

In [10]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quant_config,
    low_cpu_mem_usage=True
)

Loading weights:   0%|          | 0/453 [00:00<?, ?it/s]

/Users/aldan/Desktop/proyectos/benchmark_quant_llm/venv/lib/python3.12/site-packages/bitsandbytes/backends/default/ops.py:223: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


## Tokenization

In [11]:
inputs = tokenizer(
    PROMPT,
    return_tensors="pt"
)

inputs = {k: v.to(device) for k, v in inputs.items()}

## Benchmark

In [12]:
initial_ram = get_ram_usage_gb()

start_time = time.time()

outputs = model.generate(
    **inputs,
    max_new_tokens=MAX_NEW_TOKENS,
    do_sample=True,
    temperature=0.7,
    top_p=0.9,
    repetition_penalty=1.1
)

end_time = time.time()

final_ram = get_ram_usage_gb()

[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
/Users/aldan/Desktop/proyectos/benchmark_quant_llm/venv/lib/python3.12/site-packages/bitsandbytes/backends/cpu/ops.py:132: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


## Decode

In [13]:
generated_text = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True,
    clean_up_tokenization_spaces=False
)

print(generated_text)


You are an AI expert.

Explain quantization in large language models
for a beginner.

Use simple language and avoid code.

Hint:
1. What is the difference between
   quantized language model (MLM)
   and un-quantized language model (UNGM)?
2. How can we use the quantization to improve performance?


"""



## Metrics

In [14]:
total_time = end_time - start_time

generated_tokens = outputs.shape[1]

tokens_per_second = generated_tokens / total_time

print(f"Inference Time: {total_time:.2f} sec")
print(f"Tokens/sec: {tokens_per_second:.2f}")
print(f"RAM Usage: {(final_ram - initial_ram):.2f} GB")

Inference Time: 291.75 sec
Tokens/sec: 0.29
RAM Usage: 1.08 GB


## Save results

In [15]:
results = {
    "model": "Phi-2",
    "precision": "INT4",
    "inference_time": total_time,
    "tokens_per_second": tokens_per_second,
    "ram_usage_gb": final_ram - initial_ram
}

df = pd.DataFrame([results])

df.to_csv(
    "../results/phi2_int4.csv",
    index=False
)

print(df)

   model precision  inference_time  tokens_per_second  ram_usage_gb
0  Phi-2      INT4      291.753205            0.29477       1.08136


## Cleanup

In [16]:
del model

cleanup()